# AMPG Banten · Database Completion Rate Analysis
**PD I AMPG Provinsi Banten — Evaluasi Database & Rakorda**

Analysis of member database collection across 8 regencies/cities in Banten Province,
in preparation for the 2024 General Election.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load data
df = pd.read_csv('../data/regional_chapters.csv')
df_prov = pd.read_csv('../data/provincial_board.csv')

print('Regional chapters loaded:')
print(df[['region', 'total_sk', 'filled', 'completion_rate_pct', 'rakorda_status']].to_string(index=False))

## 1. Completion Rate per Region

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#e74c3c' if r < 50 else '#f39c12' if r < 80 else '#27ae60'
          for r in df['completion_rate_pct']]

bars = ax.barh(df['region'], df['completion_rate_pct'], color=colors, edgecolor='white', linewidth=0.5)

# Minimum threshold line
ax.axvline(x=80, color='gray', linestyle='--', linewidth=1, label='80% threshold')
ax.axvline(x=100, color='black', linestyle=':', linewidth=1, label='100% baseline')

# Labels
for bar, val in zip(bars, df['completion_rate_pct']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)

ax.set_xlabel('Completion Rate (%)', fontsize=12)
ax.set_title('Database Completion Rate by Region\nPD II AMPG Banten', fontsize=14, fontweight='bold')

red_patch = mpatches.Patch(color='#e74c3c', label='Critical (<50%)')
orange_patch = mpatches.Patch(color='#f39c12', label='At Risk (50–79%)')
green_patch = mpatches.Patch(color='#27ae60', label='On Track (≥80%)')
ax.legend(handles=[red_patch, orange_patch, green_patch], loc='lower right')

plt.tight_layout()
plt.savefig('completion_rate_by_region.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 2. Gap Analysis — Filled vs SK

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df))
width = 0.35

bars1 = ax.bar(x - width/2, df['total_sk'], width, label='SK (Official)', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, df['filled'], width, label='Filled', color='#2ecc71', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(df['region'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Number of Members')
ax.set_title('SK vs Filled — Gap Analysis by Region', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(y=50, color='red', linestyle='--', linewidth=1, label='Min. 50 (Kab/Kota standard)')

plt.tight_layout()
plt.savefig('sk_vs_filled_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 3. National Ranking Context

In [ ]:
national_ranking = pd.DataFrame({
    'rank': [1, 2, 3, 4, 5],
    'province': ['Sulawesi', 'Jawa Timur', 'Sumatera Utara', 'Jawa Barat', 'Banten'],
    'highlight': [False, False, False, False, True]
})

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#f39c12' if h else '#bdc3c7' for h in national_ranking['highlight']]

bars = ax.barh(national_ranking['province'][::-1], [5,4,3,2,1], color=colors[::-1], edgecolor='white')

for i, (bar, rank) in enumerate(zip(bars, [1,2,3,4,5][::-1])):
    ax.text(0.1, bar.get_y() + bar.get_height()/2,
            f'#{rank}', va='center', fontsize=12, fontweight='bold', color='white')

ax.set_xlim(0, 6)
ax.set_xticks([])
ax.set_title('National Database Ranking — AMPG\nBanten: Red Zone → Top 5 in 7 Days',
             fontsize=13, fontweight='bold')

ax.text(0.5, -0.12,
        'If Kota Cilegon + Kab. Tangerang complete → Banten #1 Nationally',
        transform=ax.transAxes, ha='center', fontsize=10,
        style='italic', color='#e74c3c')

plt.tight_layout()
plt.savefig('national_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 4. Summary Statistics

In [ ]:
total_sk = df['total_sk'].sum()
total_filled = df['filled'].sum()
total_gap = df['not_filled'].sum()
overall_rate = (total_filled / total_sk) * 100

bottleneck_gap = df[df['region'].isin(['Kota Cilegon', 'Kabupaten Tangerang'])]['not_filled'].sum()

print('=== AMPG Banten Database Summary ===')
print(f'Total SK (official):     {total_sk}')
print(f'Total filled:            {total_filled}')
print(f'Total gap:               {total_gap}')
print(f'Overall completion rate: {overall_rate:.1f}%')
print(f'')
print(f'Bottleneck gap (Cilegon + Kab. Tangerang): {bottleneck_gap} unfilled')
print(f'Rakorda completed: {df[df["rakorda_status"]=="Sudah"]["region"].tolist()}')
print(f'Rakorda pending:   {df[df["rakorda_status"]=="Belum"]["region"].tolist()}')